# Random Forest Ablation

# Setup

In [36]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

from preprocessing import clean_data, engineer_data
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [ ]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/random_forest.csv"

########## DEBUG ##########
DEBUG = True

########## MODEL ##########
INNER_CV = 2
OUTER_CV = 2
SCORING = 'neg_root_mean_squared_error'
MAX_FEATURES = 'sqrt'
RANDOM_STATE = 0

# Param Grid
N_ESTIMATORS = [300]
MAX_DEPTH = [15]
MIN_SAMPLES_SPLIT = [10]
MIN_SAMPLES_LEAF = [5]

# Init & Pre-Processing

In [38]:
data = pd.read_csv (TRAIN_PATH)
labels, cleaned_data = clean_data (data)
feature_engineer = FunctionTransformer (engineer_data)

# Model

In [39]:
# Pipeline
pipeline = Pipeline ([('f_eng', feature_engineer),
                      ('rf', RandomForestRegressor (
                                    max_features = MAX_FEATURES,
                                    random_state = RANDOM_STATE))])

# Grid Search Double CV
param_grid = {'rf__n_estimators': N_ESTIMATORS,
              'rf__max_depth': MAX_DEPTH,
              'rf__min_samples_split': MIN_SAMPLES_SPLIT,
              'rf__min_samples_leaf': MIN_SAMPLES_LEAF}

gs = GridSearchCV (estimator = pipeline,
                   param_grid = param_grid,
                   scoring = SCORING,
                   cv = INNER_CV)

nested_scores = cross_val_score (gs,
                                 cleaned_data,
                                 labels,
                                 cv = OUTER_CV,
                                 scoring = SCORING,
                                 n_jobs = -1)

# Evaluate
if (DEBUG):
    print (f"Nested RMSE: {-nested_scores.mean ()}")
    print (f"Fold RMSEs: {-nested_scores}")

Nested RMSE: 4.281999444790026
Fold RMSEs: [4.28378338 4.28021551]


# Predict

In [40]:
# Build final model with all training data
gs.fit (cleaned_data, labels)
if (DEBUG):
    print ("Best parameters:", gs.best_params_)
    print ("Best inner CV score:", -gs.best_score_)

final_model = gs.best_estimator_

# Predict
test_data = pd.read_csv (TEST_PATH)
_, cleaned_test_data = clean_data (test_data)

predictions = final_model.predict (cleaned_test_data)

# Save
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (predictions) + 1),
                          'Milk_Yield_L': predictions})
out_data.to_csv (OUT_PATH, index = False)

Best parameters: {'rf__max_depth': 15, 'rf__min_samples_leaf': 5, 'rf__min_samples_split': 10, 'rf__n_estimators': 400}
Best inner CV score: 4.281999444790026
